In [2]:
# ============================================================
# Task 2: Image Deblurring with NAFNet
# COMP6001 Assignment 1
# ============================================================

import time, warnings, subprocess
from pathlib import Path

import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from skimage.metrics import structural_similarity as ssim_fn
from skimage.metrics import peak_signal_noise_ratio as psnr_fn
import matplotlib.pyplot as plt
from tqdm import tqdm

warnings.filterwarnings("ignore")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


# ── 1. Paths ─────────────────────────────────────────────────
BLUR_DIR   = Path("/kaggle/input/datasets/jishnuparayilshibu/a-curated-list-of-image-deblurring-datasets/DBlur/Gopro/test/blur")
SHARP_DIR  = Path("/kaggle/input/datasets/jishnuparayilshibu/a-curated-list-of-image-deblurring-datasets/DBlur/Gopro/test/sharp")
OUTPUT_DIR = Path("/kaggle/working/outputs/task2")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MAX_SAMPLES = 10


# ── 2. NAFNet ─────────────────────────────────────────────────

class LayerNorm2d(nn.LayerNorm):
    def forward(self, x):
        return super().forward(x.permute(0,2,3,1)).permute(0,3,1,2)

class SimpleGate(nn.Module):
    def forward(self, x):
        x1, x2 = x.chunk(2, dim=1)
        return x1 * x2

class NAFBlock(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.norm1 = LayerNorm2d(c)
        self.norm2 = LayerNorm2d(c)
        self.conv1 = nn.Conv2d(c, c*2, 1)
        self.conv2 = nn.Conv2d(c*2, c*2, 3, 1, 1, groups=c*2)
        self.conv3 = nn.Conv2d(c, c, 1)
        self.sca   = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Conv2d(c, c, 1))
        self.sg    = SimpleGate()
        self.conv4 = nn.Conv2d(c, c*2, 1)
        self.conv5 = nn.Conv2d(c, c, 1)
        self.beta  = nn.Parameter(torch.ones(1, c, 1, 1))
        self.gamma = nn.Parameter(torch.ones(1, c, 1, 1))

    def forward(self, inp):
        x = self.sg(self.conv2(self.conv1(self.norm1(inp))))
        y = inp + self.conv3(x * self.sca(x)) * self.beta
        return y + self.conv5(self.sg(self.conv4(self.norm2(y)))) * self.gamma

class NAFNet(nn.Module):
    def __init__(self, width=64, enc_blks=[2,2,4,8], middle_blk_num=12, dec_blks=[2,2,2,2]):
        super().__init__()
        self.intro       = nn.Conv2d(3, width, 3, 1, 1)
        self.ending      = nn.Conv2d(width, 3, 3, 1, 1)
        self.encoders    = nn.ModuleList()
        self.downs       = nn.ModuleList()
        self.middle_blks = nn.ModuleList()
        self.ups         = nn.ModuleList()
        self.decoders    = nn.ModuleList()

        chan = width
        for n in enc_blks:
            self.encoders.append(nn.Sequential(*[NAFBlock(chan) for _ in range(n)]))
            self.downs.append(nn.Conv2d(chan, chan*2, 2, 2))
            chan *= 2

        self.middle_blks = nn.Sequential(*[NAFBlock(chan) for _ in range(middle_blk_num)])

        for n in dec_blks:
            self.ups.append(nn.Sequential(nn.Conv2d(chan, chan*2, 1, bias=False), nn.PixelShuffle(2)))
            chan //= 2
            self.decoders.append(nn.Sequential(*[NAFBlock(chan) for _ in range(n)]))

        self.padder_size = 2 ** len(self.encoders)

    def forward(self, inp):
        B, C, H, W = inp.shape
        ph = (self.padder_size - H % self.padder_size) % self.padder_size
        pw = (self.padder_size - W % self.padder_size) % self.padder_size
        x  = F.pad(inp, (0, pw, 0, ph), mode='reflect')
        x  = self.intro(x)
        encs = []
        for enc, down in zip(self.encoders, self.downs):
            x = enc(x); encs.append(x); x = down(x)
        x = self.middle_blks(x)
        for dec, up, skip in zip(self.decoders, self.ups, reversed(encs)):
            x = dec(up(x) + skip)
        return (self.ending(x) + inp)[:, :, :H, :W]


# ── 3. Load weights ───────────────────────────────────────────

def load_nafnet():
    path = Path("./weights/NAFNet-GoPro-width64.pth")
    path.parent.mkdir(exist_ok=True)

    if path.exists() and path.stat().st_size < 10 * 1024 * 1024:
        path.unlink()
        print("Removed corrupted weights file.")

    if not path.exists():
        print("Downloading NAFNet weights from Google Drive (~140 MB)…")
        subprocess.run(["pip", "install", "gdown", "-q"], check=True)
        import gdown
        gdown.download(
            id="14D4V4raNYIOhETfcuuLI3bGLB-OYIv6X",
            output=str(path),
            quiet=False
        )
        print("Download complete.")

    ckpt  = torch.load(path, map_location=device)
    state = ckpt.get("params_ema") or ckpt.get("params") or ckpt

    width      = state["intro.weight"].shape[0]
    num_stages = len({k.split(".")[1] for k in state if k.startswith("downs.")})
    def nblocks(prefix):
        idxs = {int(k[len(prefix):].split(".")[0]) for k in state
                if k.startswith(prefix) and k[len(prefix):].split(".")[0].isdigit()}
        return max(idxs) + 1 if idxs else 0
    enc_blks       = [nblocks(f"encoders.{i}.") for i in range(num_stages)]
    dec_blks       = [nblocks(f"decoders.{i}.") for i in range(num_stages)]
    middle_blk_num = nblocks("middle_blks.")

    model = NAFNet(width=width, enc_blks=enc_blks,
                   middle_blk_num=middle_blk_num, dec_blks=dec_blks)
    model_keys = set(model.state_dict().keys())
    model.load_state_dict({k: v for k, v in state.items() if k in model_keys}, strict=False)
    return model.eval().to(device)


# ── 4. Dataset ────────────────────────────────────────────────

class GoproDataset(Dataset):
    EXTS = {".png", ".jpg", ".jpeg", ".bmp"}

    def __init__(self, blur_dir, sharp_dir, max_samples=None):
        blur_dir, sharp_dir = Path(blur_dir), Path(sharp_dir)
        sharp_lookup = {
            p.relative_to(sharp_dir): p
            for p in sharp_dir.rglob("*") if p.suffix.lower() in self.EXTS
        }
        self.pairs = [
            (bp, sharp_lookup[bp.relative_to(blur_dir)])
            for bp in sorted(blur_dir.rglob("*"))
            if bp.suffix.lower() in self.EXTS and bp.relative_to(blur_dir) in sharp_lookup
        ]
        if not self.pairs:
            raise RuntimeError("No matched blur/sharp pairs found.")
        if max_samples:
            self.pairs = self.pairs[:max_samples]
        print(f"Dataset: {len(self.pairs)} pairs")

    def __len__(self): return len(self.pairs)

    def __getitem__(self, idx):
        bp, sp = self.pairs[idx]
        def read(p):
            img = cv2.imread(str(p), cv2.IMREAD_COLOR)
            return cv2.cvtColor(img, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
        sharp = read(sp)
        blur  = cv2.resize(read(bp), (sharp.shape[1], sharp.shape[0]), interpolation=cv2.INTER_AREA)
        t = lambda x: torch.from_numpy(x.transpose(2, 0, 1))
        return {"blur": t(blur), "sharp": t(sharp)}


# ── 5. Metrics ────────────────────────────────────────────────

def compute_metrics(pred, target):
    psnr = psnr_fn(target, pred, data_range=1.0)
    ssim = ssim_fn(target, pred, data_range=1.0, channel_axis=2)
    return psnr, ssim


# ── 6. Plots ──────────────────────────────────────────────────

def save_comparison(blur, sharp, deblurred, idx, save_dir, psnr_b, ssim_b, psnr_d, ssim_d):
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    fig.patch.set_facecolor("#111827")
    for ax, img, title in zip(axes,
        [blur, deblurred, sharp],
        [f"Blurred\nPSNR={psnr_b:.2f} dB  SSIM={ssim_b:.4f}",
         f"NAFNet Output\nPSNR={psnr_d:.2f} dB  SSIM={ssim_d:.4f}",
         "Sharp (Ground Truth)"]
    ):
        ax.imshow(np.clip(img, 0, 1))
        ax.set_title(title, color="white", fontsize=10)
        ax.axis("off")
    plt.tight_layout(pad=1.5)
    plt.savefig(save_dir / f"comparison_{idx:04d}.png", dpi=150,
                bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.close(fig)

def save_metric_plot(results):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle("NAFNet — PSNR & SSIM", fontsize=13, fontweight="bold")
    for ax, bk, dk, label in zip(axes,
        ["psnr_blur", "ssim_blur"],
        ["psnr_deblurred", "ssim_deblurred"],
        ["PSNR (dB)", "SSIM"]
    ):
        b, d, x = [r[bk] for r in results], [r[dk] for r in results], np.arange(len(results))
        ax.plot(x, b, color="#6c88d4", label="Blurred")
        ax.plot(x, d, color="#e88b3a", label="Deblurred")
        ax.fill_between(x, b, d, where=[dv>=bv for dv,bv in zip(d,b)], alpha=0.15, color="#e88b3a")
        ax.set_ylabel(label); ax.set_xlabel("Image index")
        ax.legend(); ax.grid(linewidth=0.4, alpha=0.5)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "metric_summary.png", dpi=150, bbox_inches="tight")
    plt.close(fig)


# ── 7. Main ───────────────────────────────────────────────────

def run_task2(max_samples=MAX_SAMPLES, visualise_every=2):
    loader  = DataLoader(GoproDataset(BLUR_DIR, SHARP_DIR, max_samples),
                         batch_size=1, shuffle=False, num_workers=2)
    model   = load_nafnet()
    vis_dir = OUTPUT_DIR / "comparisons"
    vis_dir.mkdir(exist_ok=True)
    results = []

    for i, batch in enumerate(tqdm(loader, desc="Deblurring")):
        blur_t, sharp_t = batch["blur"][0], batch["sharp"][0]

        t0 = time.perf_counter()
        with torch.inference_mode():
            deblu_t = model(blur_t.unsqueeze(0).to(device)).clamp(0,1)[0].cpu()
        elapsed = time.perf_counter() - t0

        np3     = lambda t: t.numpy().transpose(1,2,0)
        blur_np = np3(blur_t)
        shp_np  = np3(sharp_t)
        deb_np  = np3(deblu_t)

        psnr_b, ssim_b = compute_metrics(blur_np, shp_np)
        psnr_d, ssim_d = compute_metrics(deb_np,  shp_np)

        results.append({
            "idx":            i,
            "psnr_blur":      psnr_b, "psnr_deblurred": psnr_d,
            "ssim_blur":      ssim_b, "ssim_deblurred": ssim_d,
            "psnr_gain":      psnr_d - psnr_b,
            "ssim_gain":      ssim_d - ssim_b,
            "latency_s":      elapsed,
        })

        if i % visualise_every == 0:
            save_comparison(blur_np, shp_np, deb_np, i, vis_dir,
                            psnr_b, ssim_b, psnr_d, ssim_d)

        cv2.imwrite(str(OUTPUT_DIR / f"deblurred_{i:04d}.png"),
                    cv2.cvtColor((deb_np*255).astype(np.uint8), cv2.COLOR_RGB2BGR))

    n   = len(results)
    avg = lambda k: sum(r[k] for r in results) / n

    print(f"\n{'='*57}")
    print(f"  NAFNet Results — {n} images")
    print(f"{'='*57}")
    print(f"  {'Metric':<24}  {'Blurred':>10}  {'Deblurred':>10}  {'Gain':>8}")
    print(f"  {'-'*53}")
    print(f"  {'PSNR (dB)':<24}  {avg('psnr_blur'):>10.4f}  {avg('psnr_deblurred'):>10.4f}  {avg('psnr_gain'):>+8.4f}")
    print(f"  {'SSIM':<24}  {avg('ssim_blur'):>10.4f}  {avg('ssim_deblurred'):>10.4f}  {avg('ssim_gain'):>+8.4f}")
    print(f"  {'Avg latency (s)':<24}  {avg('latency_s'):>10.4f}")
    print(f"{'='*57}\n")
    print(f"  {'idx':>4}  {'PSNR_blur':>9}  {'PSNR_deblu':>10}  {'SSIM_blur':>9}  {'SSIM_deblu':>10}")
    for r in results:
        print(f"  {r['idx']:>4}  {r['psnr_blur']:>9.3f}  {r['psnr_deblurred']:>10.3f}"
              f"  {r['ssim_blur']:>9.4f}  {r['ssim_deblurred']:>10.4f}")

    save_metric_plot(results)
    return results


results = run_task2()

Using device: cuda
Dataset: 10 pairs


Downloading...
From (original): https://drive.google.com/uc?id=14D4V4raNYIOhETfcuuLI3bGLB-OYIv6X
From (redirected): https://drive.google.com/uc?id=14D4V4raNYIOhETfcuuLI3bGLB-OYIv6X&confirm=t&uuid=eb069098-ac16-456f-a36e-9a5ff0459c9c
To: /kaggle/working/weights/NAFNet-GoPro-width64.pth
100%|██████████| 272M/272M [00:02<00:00, 127MB/s] 


Download complete.


Deblurring: 100%|██████████| 10/10 [00:21<00:00,  2.14s/it]



  NAFNet Results — 10 images
  Metric                       Blurred   Deblurred      Gain
  -----------------------------------------------------
  PSNR (dB)                    26.4127     29.3281   +2.9154
  SSIM                          0.8625      0.9137   +0.0512
  Avg latency (s)               1.1686

   idx  PSNR_blur  PSNR_deblu  SSIM_blur  SSIM_deblu
     0     20.362      25.502     0.7194      0.8553
     1     22.015      28.300     0.7552      0.9018
     2     24.300      28.831     0.8334      0.9174
     3     26.827      29.265     0.8264      0.8756
     4     25.516      29.008     0.8373      0.9102
     5     31.035      31.700     0.9565      0.9491
     6     27.896      29.832     0.9142      0.9283
     7     28.049      29.853     0.9169      0.9282
     8     28.591      30.190     0.9234      0.9301
     9     29.536      30.801     0.9419      0.9408
